# WRO 2026 Future Engineers - Red/Green Traffic-Sign Detector

Trains a **YOLO11n** detector (PyTorch) on a **synthetically generated** dataset - no manual labeling -
then exports to **NCNN + TFLite** for the Raspberry Pi 5.

Classes: `0 = red`, `1 = green`. Official colors: red RGB (238,39,55), green RGB (68,214,44), signs 50x50x100 mm.

**Runtime -> Change runtime type -> T4 GPU** before running. Then Runtime -> Run all.

> Synthetic data gets you a working model in minutes. For competition accuracy, shoot ~150 real
> frames from the car's camera, label them (Roboflow / LabelImg), and re-run training on that folder.

In [ ]:
# 1. Install
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 2. Synthetic data generator (domain-randomized). Labels are exact by construction.
import cv2, numpy as np, random

RED_BGR=(55,39,238); GREEN_BGR=(44,214,68); MAGENTA=(255,0,255)   # official colors, BGR
def jit(c,a=22): return int(np.clip(c+random.randint(-a,a),0,255))

def bg(W,H):
    top,bot=random.randint(175,255),random.randint(175,255)
    col=np.linspace(top,bot,H).reshape(H,1,1)
    img=np.repeat(np.repeat(col,W,1),3,2).astype(np.uint8)
    # black walls (100 mm): top band always, sides sometimes
    img[:random.randint(15,90),:]=random.randint(0,45)
    if random.random()<0.4: img[:, :random.randint(15,70)]=random.randint(0,45)
    if random.random()<0.4: img[:, W-random.randint(15,70):]=random.randint(0,45)
    return img

def draw_pillar(img,color_bgr):
    H,W=img.shape[:2]
    pw=random.randint(12,72); ph=int(pw*random.uniform(1.7,2.3))
    x=random.randint(0,W-pw); y=random.randint(int(H*0.12),max(int(H*0.12)+1,H-ph))
    col=np.array([jit(c) for c in color_bgr],float)
    grad=np.linspace(1.15,0.72,ph).reshape(ph,1,1)          # top-lit shading
    block=np.clip(col.reshape(1,1,3)*grad,0,255).astype(np.uint8)
    img[y:y+ph,x:x+pw]=np.repeat(block,pw,1)
    return x,y,pw,ph

def photometric(img):
    img=cv2.convertScaleAbs(img,alpha=random.uniform(0.6,1.4),beta=random.randint(-30,30))
    if random.random()<0.5:
        k=random.choice([3,5]); img=cv2.GaussianBlur(img,(k,k),0)
    if random.random()<0.5:
        n=np.random.normal(0,7,img.shape).astype(np.int16)
        img=np.clip(img.astype(np.int16)+n,0,255).astype(np.uint8)
    return img

def make_image(W=640,H=480):
    img=bg(W,H); labels=[]
    for _ in range(random.randint(1,5)):
        cls=random.randint(0,1)
        x,y,pw,ph=draw_pillar(img,RED_BGR if cls==0 else GREEN_BGR)
        labels.append((cls,(x+pw/2)/W,(y+ph/2)/H,pw/W,ph/H))
    for _ in range(random.randint(0,2)):                    # magenta parking bar (unlabeled distractor)
        x=random.randint(0,W-60); y=random.randint(0,H-16)
        cv2.rectangle(img,(x,y),(x+random.randint(30,90),y+random.randint(6,16)),
                      tuple(jit(c) for c in MAGENTA),-1)
    return photometric(img),labels

print("generator ready"); print("example labels:",make_image()[1][:2])

In [ ]:
# 3. Build the dataset in YOLO format
from pathlib import Path
ROOT=Path('/content/dataset')
for s in ('train','val'):
    (ROOT/'images'/s).mkdir(parents=True,exist_ok=True)
    (ROOT/'labels'/s).mkdir(parents=True,exist_ok=True)

def build(split,n):
    for i in range(n):
        img,labels=make_image()
        cv2.imwrite(str(ROOT/'images'/split/f'{i:05d}.jpg'),img)
        with open(ROOT/'labels'/split/f'{i:05d}.txt','w') as f:
            for cls,xc,yc,w,h in labels:
                f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

random.seed(0); np.random.seed(0)
build('train',1200); build('val',200)
open(ROOT/'data.yaml','w').write(f"""path: {ROOT}
train: images/train
val: images/val
names:
  0: red
  1: green
""")
print('dataset ready at',ROOT)

In [ ]:
# 4. Sanity-check: draw the labels on a few training images
import matplotlib.pyplot as plt, glob
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,fp in zip(axes,random.sample(glob.glob(str(ROOT/'images/train/*.jpg')),3)):
    img=cv2.cvtColor(cv2.imread(fp),cv2.COLOR_BGR2RGB); H,W=img.shape[:2]
    for line in open(fp.replace('images','labels').replace('.jpg','.txt')):
        cls,xc,yc,w,h=map(float,line.split())
        p1=(int((xc-w/2)*W),int((yc-h/2)*H)); p2=(int((xc+w/2)*W),int((yc+h/2)*H))
        cv2.rectangle(img,p1,p2,(255,0,0) if cls==0 else (0,160,0),2)
    ax.imshow(img); ax.axis('off')
plt.show()

In [ ]:
# 5. Train YOLO11n. hue augmentation kept SMALL - color is the whole game here.
from ultralytics import YOLO
model=YOLO('yolo11n.pt')
model.train(
    data=str(ROOT/'data.yaml'),
    epochs=40, imgsz=640, batch=32,
    hsv_h=0.01, hsv_s=0.5, hsv_v=0.4,   # don't let red drift toward orange/green
    fliplr=0.5, mosaic=1.0, degrees=5,
    project='wro', name='signs', exist_ok=True)

In [ ]:
# 6. Metrics + predictions on the val set
m=model.val()
print(f"mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}")
res=model.predict(str(ROOT/'images/val'),conf=0.5,max_det=10,verbose=False)
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,r in zip(axes,res[:3]):
    ax.imshow(cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.show()

In [ ]:
# 7. Export for the Raspberry Pi 5
#    NCNN   -> fastest on Pi CPU (Ultralytics-recommended)
#    TFLite -> TensorFlow runtime, int8 quantized (smallest)
best=YOLO('wro/signs/weights/best.pt')
best.export(format='ncnn')                                            # -> best_ncnn_model/
best.export(format='tflite', int8=True, data=str(ROOT/'data.yaml'))  # -> *_full_integer_quant.tflite
print('done - check wro/signs/weights/')

In [ ]:
# 8. Test on YOUR OWN photo (upload a frame from the car's camera)
from google.colab import files
up=files.upload()
for fn in up:
    r=model.predict(fn,conf=0.4,verbose=False)[0]
    plt.figure(figsize=(8,6)); plt.imshow(cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
    for b in r.boxes:
        print(model.names[int(b.cls)], f"conf={float(b.conf):.2f}", "xywh=",[round(v) for v in b.xywh[0].tolist()])

In [ ]:
# 9. Download the trained weights + exports
from google.colab import files
!zip -qr /content/wro_signs.zip wro/signs/weights
files.download('/content/wro_signs.zip')

## Deploying on the Pi 5

1. Unzip; copy `best_ncnn_model/` (or the `*_full_integer_quant.tflite`) to the robot.
2. `pip install ultralytics` on the Pi, then load with `YOLO('best_ncnn_model', task='detect')`.
3. Use the `MLBlockDetector` drop-in (given in chat) - it returns the same `Detection` objects as the
   HSV detector, so it plugs straight into your ROS node and the LiDAR-range fusion is unchanged.

### Closing the synthetic->real gap (do this before the competition)
- Record ~150 frames from the actual camera on the actual field (varied lighting, angles, distances).
- Label them (Roboflow is fastest), keep the same `0=red, 1=green` classes.
- Re-run cell 5 with `data=` pointing at the real dataset, starting from `best.pt` (or mix synthetic+real).
- Lock the camera's **auto-exposure and auto-white-balance** so inference matches training.

### Speed on Pi 5
- Use `imgsz=320` at inference for ~2x speed; the pillars are big and solid, so accuracy barely drops.
- NCNN on 4 threads gets you real-time; TFLite int8 is close. Benchmark both with `yolo benchmark`.